# Single-Agent Smart Assistant — Agent Pipeline Project

## Problem Statement
This notebook builds a **single-agent smart assistant** that can understand a user's query, decide which tool (if any) fits the request, run that tool, and always return a clean, structured response — regardless of what path it took internally.

The agent supports three intents:
- **Math queries** → routed to a Calculator Tool
- **Keyword extraction requests** → routed to a Keyword Extractor Tool
- **Everything else** → handled as a general/direct response

### What's implemented here
- Two core tools (Calculator, Keyword Extractor) plus one bonus tool (Word Counter)
- Conditional routing logic in the agent function
- A lightweight **state log** that tracks which "node" (step) the agent visited for each query — this reflects the stateful directed graph idea from the quiz (nodes = steps, edges = the routing decisions between them)
- A schema check so every response follows the same `{"type": ..., "result": ...}` structure
- Error handling with try/except around every tool call
- Test cases + an interactive mode to try it live


## Tool 1 — Calculator

Rather than using raw `eval()` on the whole expression (which can break on stray words and is unsafe for arbitrary input), this version:
1. Strips out anything that isn't a digit, operator, decimal point, or parenthesis
2. Evaluates only the cleaned numeric expression

This makes the tool a bit more robust to messy input like `"calculate 20 + 5 please"`.


In [19]:
import re

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression safely.

    Strips non-math characters first so trigger words like 'calculate'
    or trailing punctuation don't break the evaluation.
    """
    try:
        cleaned = re.sub(r"[^0-9+\-*/%.() ]", "", expression)
        cleaned = cleaned.strip()
        if not cleaned:
            return "Error in calculation"
        return str(eval(cleaned, {"__builtins__": {}}, {}))
    except Exception:
        return "Error in calculation"

## Tool 2 — Keyword Extractor

Improved slightly over a plain "any word longer than 4 letters" filter by ignoring a small
set of common stopwords, so filler words like "about" or "which" don't crowd out real keywords.


In [20]:
STOPWORDS = {"about", "which", "there", "their", "would", "these", "those", "extract", "keywords", "from"}

def extract_keywords(text: str) -> list:
    """Extract up to 5 keywords from text, ignoring common stopwords."""
    try:
        words = text.split()
        keywords = [w.lower().strip(".,!?") for w in words if len(w) > 4]
        keywords = [w for w in keywords if w not in STOPWORDS]
        seen = set()
        unique_keywords = []
        for w in keywords:
            if w not in seen:
                seen.add(w)
                unique_keywords.append(w)
        return unique_keywords[:5]
    except Exception:
        return []


## Bonus Tool — Word Counter

A small third tool to demonstrate the agent can scale to more than two intents without
changing its core structure — just one more routing branch and one more node.


In [21]:
def word_counter(text: str) -> int:
    """Count the number of words in the given text."""
    try:
        return len(text.split())
    except Exception:
        return 0


## Agent Logic — Routing, State, and Schema Validation

The `agent()` function is the core of this project. It does four things every time it runs:

1. **Routes** the query to the right tool based on keywords in the text (conditional routing)
2. **Calls** that tool, wrapped in error handling so a tool failure never crashes the agent
3. **Logs** which node it visited into `execution_log` — a simple stand-in for the "state" that
   would be tracked across a stateful directed graph
4. **Validates** the output against the expected schema before returning it, so the response
   shape is always `{"type": ..., "result": ...}`


In [22]:
exec_log = []  # tracks the path the agent has taken across all queries so far

def looks_like_math(query: str) -> bool:
    """Check if a query is purely a math expression (digits/operators only),
    even if it doesn't contain the word 'calculate'.
    """
    stripped = query.strip().strip('"').strip("'")
    return bool(re.fullmatch(r"[0-9+\-*/%.() ]+", stripped)) and stripped != ""


def validate_schema(response: dict) -> bool:
    """Confirm the response matches the expected {type, result} schema."""
    return isinstance(response, dict) and "type" in response and "result" in response


def agent(query: str) -> dict:
    query_lower = query.lower()
    node_visited = None

    try:
        if "calculate" in query_lower or looks_like_math(query):
            node_visited = "calculator_tool"
            expression = query_lower.replace("calculate", "").strip().strip('"').strip("'")
            result = calculator(expression)
            response = (
                {"type": "error", "result": result}
                if result == "Error in calculation"
                else {"type": "calculation", "result": result}
            )

        elif "keywords" in query_lower:
            node_visited = "keyword_tool"
            text = query_lower.replace("extract keywords from", "").strip()
            response = {"type": "keywords", "result": extract_keywords(text)}

        elif "count words" in query_lower or "word count" in query_lower:
            node_visited = "word_counter_tool"
            text = query_lower.replace("count words in", "").replace("word count of", "").strip()
            response = {"type": "word_count", "result": word_counter(text)}

        else:
            node_visited = "general_response"
            response = {
                "type": "general",
                "result": f"You asked: '{query}'. This is a general query with no specific tool.",
            }

    except Exception as e:
        node_visited = "error_handler"
        response = {"type": "error", "result": f"Something went wrong: {str(e)}"}

    exec_log.append({"query": query, "node": node_visited})

    if not validate_schema(response):
        response = {"type": "error", "result": "Response did not match expected schema"}

    return response

## Expected Output Format

Every call to `agent()` returns:
```
{
  "type": "calculation / keywords / word_count / general / error",
  "result": ...
}
```


## Test Cases

In [23]:
queries = [
    "Calculate 125 - 35",
    "Extract keywords from Large Language Models are transforming artificial intelligence",
    "Count words in Python is one of the most popular programming languages",
    "Explain what data science is.",
    "75/5+12",
    "100-20*3+15%4"
]

for query in queries:
    print("Query:", query)
    print("Response:", agent(query))
    print("-" * 50)

Query: Calculate 125 - 35
Response: {'type': 'calculation', 'result': '90'}
--------------------------------------------------
Query: Extract keywords from Large Language Models are transforming artificial intelligence
Response: {'type': 'keywords', 'result': ['large', 'language', 'models', 'transforming', 'artificial']}
--------------------------------------------------
Query: Count words in Python is one of the most popular programming languages
Response: {'type': 'word_count', 'result': 9}
--------------------------------------------------
Query: Explain what data science is.
Response: {'type': 'general', 'result': "You asked: 'Explain what data science is.'. This is a general query with no specific tool."}
--------------------------------------------------
Query: 75/5+12
Response: {'type': 'calculation', 'result': '27.0'}
--------------------------------------------------
Query: 100-20*3+15%4
Response: {'type': 'calculation', 'result': '43'}
----------------------------------------

## Viewing the Agent's State (Execution Log)

This prints the sequence of nodes visited across all queries run so far — a small,
concrete example of state being carried and accumulated across calls, tying back to
the stateful directed graph concept from the quiz.


In [24]:
for entry in exec_log:
    print(entry)


{'query': 'Calculate 125 - 35', 'node': 'calculator_tool'}
{'query': 'Extract keywords from Large Language Models are transforming artificial intelligence', 'node': 'keyword_tool'}
{'query': 'Count words in Python is one of the most popular programming languages', 'node': 'word_counter_tool'}
{'query': 'Explain what data science is.', 'node': 'general_response'}
{'query': '75/5+12', 'node': 'calculator_tool'}
{'query': '100-20*3+15%4', 'node': 'calculator_tool'}


## Interactive Mode

Run this cell to try your own queries. Type `exit` to stop.

In [25]:
while True:
    userinput = input("Enter query (type 'exit' to stop): ")
    if userinput.lower() == "exit":
        break
    print("Response:", agent(userinput))


Enter query (type 'exit' to stop): Calculate 45 + 55
Response: {'type': 'calculation', 'result': '100'}
Enter query (type 'exit' to stop): 100-25*2
Response: {'type': 'calculation', 'result': '50'}
Enter query (type 'exit' to stop): Calculate (20+5)*4
Response: {'type': 'calculation', 'result': '100'}
Enter query (type 'exit' to stop): Extract keywords from Machine Learning models predict future values
Response: {'type': 'keywords', 'result': ['machine', 'learning', 'models', 'predict', 'future']}
Enter query (type 'exit' to stop): exit


# Agent Pipeline Visualization

This section visualizes how the Single-Agent Smart Assistant processes a user's query.

The execution flow is:

1. User enters a query.
2. The agent detects the intent.
3. Conditional routing selects the appropriate tool.
4. The selected tool performs the required task.
5. The final response is returned in JSON format.

This helps us understand the internal workflow of the agent.

In [12]:
# ==========================
# Agent Pipeline Visualization
# ==========================

def show_pipeline(query):

    response = agent(query)

    print("\n==============================")
    print("        AGENT PIPELINE")
    print("==============================")
    print("User Query")
    print("    │")
    print("    ▼")
    print("Intent Detection")
    print("    │")
    print("    ▼")
    print("Conditional Routing")
    print("    │")
    print(f"    └──► Selected Tool : {response.get('type')}")
    print("             │")
    print("             ▼")
    print("      Tool Execution")
    print("             │")
    print("             ▼")
    print("      JSON Response")
    print("==============================")

    print(response)

show_pipeline("Calculate 45 + 10")


        AGENT PIPELINE
User Query
    │
    ▼
Intent Detection
    │
    ▼
Conditional Routing
    │
    └──► Selected Tool : calculation
             │
             ▼
      Tool Execution
             │
             ▼
      JSON Response
{'type': 'calculation', 'result': '55'}


# Execution History

The execution history records every query processed by the agent.

This information is useful for:

- Tracking executed queries
- Debugging the workflow
- Understanding tool usage
- Evaluating agent performance

In [14]:
# ==========================
# Execution History
# ==========================

def show_execution_history():

    print("\n========== EXECUTION HISTORY ==========\n")

    if len(exec_log) == 0:
        print("No executions yet.")
        return

    for i, log in enumerate(exec_log, start=1):

        print(f"Execution {i}")

        for key, value in log.items():
            print(f"{key} : {value}")

        print("-"*40)

show_execution_history()


========== EXECUTION HISTORY ==========

Execution 1
query : Calculate 125 - 35
node : calculator_tool
----------------------------------------
Execution 2
query : Extract keywords from Large Language Models are transforming artificial intelligence
node : keyword_tool
----------------------------------------
Execution 3
query : Count words in Python is one of the most popular programming languages
node : word_counter_tool
----------------------------------------
Execution 4
query : Explain what data science is.
node : general_response
----------------------------------------
Execution 5
query : 75/5+12
node : calculator_tool
----------------------------------------
Execution 6
query : 100-20*3+15%4
node : calculator_tool
----------------------------------------
Execution 7
query : Hello?
node : general_response
----------------------------------------
Execution 8
query : Calculate 45 + 55
node : calculator_tool
----------------------------------------
Execution 9
query : Calculate (20

# Agent Statistics

This section summarizes how the agent has been used.

The statistics include:

- Calculator tool usage
- Keyword extraction usage
- Word counter usage
- General responses
- Total number of processed queries

These metrics help evaluate the efficiency of the agent.

In [15]:
# ==========================
# Agent Statistics
# ==========================

def agent_statistics():

    calculator = 0
    keywords = 0
    word_counter = 0
    general = 0

    for log in exec_log:

        node = str(log.get("node", "")).lower()

        if "calculator" in node:
            calculator += 1

        elif "keyword" in node:
            keywords += 1

        elif "word" in node:
            word_counter += 1

        else:
            general += 1

    print("\n========== AGENT STATISTICS ==========\n")

    print("Calculator Tool Used :", calculator)
    print("Keyword Tool Used    :", keywords)
    print("Word Counter Used    :", word_counter)
    print("General Responses    :", general)
    print("Total Queries        :", len(exec_log))

agent_statistics()


========== AGENT STATISTICS ==========

Calculator Tool Used : 8
Keyword Tool Used    : 2
Word Counter Used    : 1
General Responses    : 2
Total Queries        : 13


# Additional Test Cases

The following test cases verify that the routing logic works correctly.

The agent is tested with:

- Mathematical calculations
- Keyword extraction
- Word counting
- General questions
- Direct mathematical expressions

These tests ensure the reliability of the agent.

In [16]:
# ==========================
# Additional Test Cases
# ==========================

test_queries = [

    "Calculate 100/5",

    "Calculate 20*(5+3)",

    "Extract keywords from LangGraph enables stateful agent workflows",

    "Count words in Artificial Intelligence is transforming the software industry",

    "Who invented Python?",

    "What is Deep Learning?",

    "50+80"

]

for q in test_queries:

    print("\nQuery :", q)

    print(agent(q))

    print("-"*60)


Query : Calculate 100/5
{'type': 'calculation', 'result': '20.0'}
------------------------------------------------------------

Query : Calculate 20*(5+3)
{'type': 'calculation', 'result': '160'}
------------------------------------------------------------

Query : Extract keywords from LangGraph enables stateful agent workflows
{'type': 'keywords', 'result': ['langgraph', 'enables', 'stateful', 'agent', 'workflows']}
------------------------------------------------------------

Query : Count words in Artificial Intelligence is transforming the software industry
{'type': 'word_count', 'result': 7}
------------------------------------------------------------

Query : Who invented Python?
{'type': 'general', 'result': "You asked: 'Who invented Python?'. This is a general query with no specific tool."}
------------------------------------------------------------

Query : What is Deep Learning?
{'type': 'general', 'result': "You asked: 'What is Deep Learning?'. This is a general query wit

# Agent Summary Dashboard

This dashboard provides a quick overview of the current state of the agent.

It displays:

- Total executions
- Most recent execution
- Current status

This acts as a simple monitoring dashboard for the Smart Assistant.

In [18]:
# ==========================
# Agent Summary Dashboard
# ==========================

print("\n===================================")
print("     SINGLE AGENT DASHBOARD")
print("===================================")

print("Total Executions :", len(exec_log))

if exec_log:

    print("\nLast Execution")

    for key, value in exec_log[-1].items():

        print(f"{key} : {value}")

print("\nStatus : Agent Running Successfully")
print("===================================")


     SINGLE AGENT DASHBOARD
Total Executions : 20

Last Execution
query : 50+80
node : calculator_tool

Status : Agent Running Successfully
